### 📝 Task info:

### 💻 Code Work:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 📤 0. Data Loading (from db load tables data as pandas df)

In [4]:
!pip install mysql-connector-python
import pandas as pd
import mysql.connector

# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="ABCD",
    database="bank_marketing"
)

# Load validated table data into Pandas DataFrame
df = pd.read_sql("SELECT * FROM bank_marketing", conn)


C:\Users\Dell\AppData\Local\Temp\ipykernel_10812\3938222366.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM bank_marketing", conn)


### 👩🏻‍💻 1. Modeling

#### 🛠️ 1.1 Pre-Requisites


In [5]:
# Select numerical features for clustering
cluster_features = [
    'age',
    'balance',
    'day',
    'duration_minutes',
    'campaign',
    'previous',
    'pdays'
]

# Entire clustering data is considered as X
X = df[cluster_features]

X.head()

,age,balance,day,duration_minutes,campaign,previous,pdays
0,58,2143.0,5,4.35,1,0,-1
1,44,29.0,5,2.52,1,0,-1
2,33,2.0,5,1.27,1,0,-1
3,47,1506.0,5,1.53,1,0,-1
4,33,1.0,5,3.30,1,0,-1


In [6]:
X.shape

(45211, 7)

In [7]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               45211 non-null  int64  
 1   balance           45211 non-null  float64
 2   day               45211 non-null  int64  
 3   duration_minutes  45211 non-null  float64
 4   campaign          45211 non-null  int64  
 5   previous          45211 non-null  int64  
 6   pdays             45211 non-null  int64  
dtypes: float64(2), int64(5)
memory usage: 2.4 MB


In [8]:
X.isnull().sum()

age                 0
balance             0
day                 0
duration_minutes    0
campaign            0
previous            0
pdays               0
dtype: int64

In [9]:
Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_counts = ((X < lower_bound) | (X > upper_bound)).sum()

outlier_counts

age                  487
balance             4729
day                    0
duration_minutes    3235
campaign            3064
previous            8257
pdays               8257
dtype: int64

### 
- Outlier Handling: Outliers were identified using the IQR method. Outliers were found in age, balance, duration_minutes, campaign, previous, and pdays, while day had no detected outliers. Since these observations represent genuine customer characteristics and campaign behavior, they were retained for clustering rather than removed.

#### ⚙️ 1.1.2 Feature Engineering

- **Feature Selection (Mandatory) :**

In [15]:
cluster_features = [
    'age',
    'balance',
    'day',
    'duration_minutes',
    'campaign',
    'previous',
    'pdays'
]

X = df[cluster_features]

###
- **Feature Generation**: No additional features were generated because the selected numerical variables were sufficient for customer segmentation.

- **Feature Modification — Encoding** encoding: Encoding was not required because all selected clustering features are numerical. Categorical variables were excluded from the K-Means model

- **Feature Modification — Scaling  Mandatory**

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled.shape

(45211, 7)

In [17]:
X_scaled[:5]

array([[ 1.60696496,  0.25641925, -1.29847633,  0.01101364, -0.56935064,
        -0.25194037, -0.41145311],
       [ 0.28852927, -0.43789469, -1.29847633, -0.41535258, -0.56935064,
        -0.25194037, -0.41145311],
       [-0.74738448, -0.44676247, -1.29847633, -0.70658633, -0.56935064,
        -0.25194037, -0.41145311],
       [ 0.5710512 ,  0.04720545, -1.29847633, -0.64600971, -0.56935064,
        -0.25194037, -0.41145311],
       [-0.74738448, -0.44709091, -1.29847633, -0.23362272, -0.56935064,
        -0.25194037, -0.41145311]])

### 🤖 1.2 Model + Define, Train & Study

In [18]:
from sklearn.cluster import KMeans

In [19]:
kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

In [20]:
cluster_labels = kmeans.fit_predict(X_scaled)

In [21]:
X_cluster = X.copy()

X_cluster['Cluster'] = cluster_labels

X_cluster.head()

,age,balance,day,duration_minutes,campaign,previous,pdays,Cluster
0,58,2143.0,5,4.35,1,0,-1,1
1,44,29.0,5,2.52,1,0,-1,1
2,33,2.0,5,1.27,1,0,-1,1
3,47,1506.0,5,1.53,1,0,-1,1
4,33,1.0,5,3.30,1,0,-1,1


In [22]:
cluster_summary = X_cluster.groupby('Cluster').mean().round(2)

cluster_summary

,age,balance,day,duration_minutes,campaign,previous,pdays
Cluster,,,,,,,
0,40.53,1437.53,13.83,3.90,2.12,3.54,259.95
1,41.31,1129.67,8.43,3.41,2.21,0.09,2.80
2,40.69,1480.84,22.89,3.19,2.55,0.08,3.88
3,41.28,1791.23,15.82,15.24,2.41,0.24,15.49
4,40.61,1144.60,21.95,2.39,15.84,0.02,0.05


| Cluster | Main characteristics                                                       | Business interpretation                                  |
| ------- | -------------------------------------------------------------------------- | -------------------------------------------------------- |
| **0**   | Highest `previous` (3.54) and `pdays` (259.95)                             | **Previously contacted customers**                       |
| **1**   | Low `previous` (0.09), very low `pdays` (2.80), moderate campaign activity | **Less previously contacted customers**                  |
| **2**   | Higher `balance` (1480.84), higher `day` (22.89), moderate campaign        | **Higher-balance customers with later campaign contact** |
| **3**   | Highest `balance` (1791.23) and **very high duration** (15.24 min)         | **Highly engaged / high-value customers**                |
| **4**   | Extremely high `campaign` (15.84), very low `previous` (0.02)              | **Frequently contacted customers**                       |


In [23]:
cluster_counts = X_cluster['Cluster'].value_counts().sort_index()

cluster_counts

Cluster
0     6339
1    16331
2    17563
3     3592
4     1386
Name: count, dtype: int64

### 
- **Cluster Distribution**: The K-Means model segmented 45,211 customers into five clusters. Cluster 2 contains the largest number of customers (17,563), followed by Cluster 1 (16,331). Cluster 4 is the smallest segment with 1,386 customers.

####  **Silhouette Score**

In [27]:
from sklearn.metrics import silhouette_score

# Take a random sample of 5,000 rows
sample_size = 5000

sample_indices = X_scaled.shape[0]

In [28]:
from sklearn.metrics import silhouette_score
import numpy as np

np.random.seed(42)

sample_indices = np.random.choice(
    X_scaled.shape[0],
    size=5000,
    replace=False
)

X_sample = X_scaled[sample_indices]
labels_sample = cluster_labels[sample_indices]

silhouette_avg = silhouette_score(
    X_sample,
    labels_sample
)

print("Silhouette Score:", silhouette_avg)

Silhouette Score: 0.21213739872285484


- **Clustering Evaluation**: The K-Means model was evaluated using the Silhouette Score. A representative sample of 5,000 observations was used to reduce computation time. The model achieved a Silhouette Score of 0.2121, indicating limited-to-moderate separation between the customer clusters. The five clusters can still provide useful customer segmentation based on financial and campaign-related characteristics.

#### ✅ 1.3 Adding predictions to Original Data

In [29]:
df['Cluster'] = cluster_labels

In [30]:
df.head()

,age,job,marital,education,default_status,balance,housing,loan,contact,day,month,duration_minutes,campaign,previous,poutcome,y,pdays,Cluster
0,58,management,married,tertiary,no,2143.0,yes,no,unknown,5,may,4.35,1,0,unknown,no,-1,1
1,44,technician,single,secondary,no,29.0,yes,no,unknown,5,may,2.52,1,0,unknown,no,-1,1
2,33,entrepreneur,married,secondary,no,2.0,yes,yes,unknown,5,may,1.27,1,0,unknown,no,-1,1
3,47,blue-collar,married,unknown,no,1506.0,yes,no,unknown,5,may,1.53,1,0,unknown,no,-1,1
4,33,unknown,single,unknown,no,1.0,no,no,unknown,5,may,3.30,1,0,unknown,no,-1,1


In [31]:
df['Cluster'].value_counts().sort_index()

Cluster
0     6339
1    16331
2    17563
3     3592
4     1386
Name: count, dtype: int64

| Cluster   |    Records |
| --------- | ---------: |
| 0         |      6,339 |
| 1         |     16,331 |
| 2         |     17,563 |
| 3         |      3,592 |
| 4         |      1,386 |
| **Total** | **45,211** |


- **Adding Predictions to Original Data**: The cluster predictions generated by the K-Means model were added to the original dataset as a new Cluster column. All 45,211 customer records were successfully assigned to one of the five clusters, enabling further customer segmentation and business analysis.

#### 📥 1.4 Saving groups & Study

In [32]:
df.to_csv("bank_marketing_clustered.csv", index=False)

In [33]:
df.to_excel("bank_marketing_clustered.xlsx", index=False)

### Filter rows based on Cluster

In [34]:
cluster_0 = df[df['Cluster'] == 0]

cluster_0.head()

,age,job,marital,education,default_status,balance,housing,loan,contact,day,month,duration_minutes,campaign,previous,poutcome,y,pdays,Cluster
24060,33,admin.,married,tertiary,no,882.0,no,no,telephone,21,oct,0.65,1,3,failure,no,151,0
24062,42,admin.,single,secondary,no,-247.0,yes,yes,telephone,21,oct,8.65,1,1,other,yes,166,0
24064,33,services,married,secondary,no,3444.0,yes,no,telephone,21,oct,2.40,1,4,failure,yes,91,0
24077,36,management,married,tertiary,no,0.0,yes,no,telephone,23,oct,2.33,1,3,failure,yes,143,0
24080,56,technician,married,secondary,no,589.0,yes,no,unknown,23,oct,8.63,1,2,success,yes,147,0


In [40]:
group_stats = df.groupby('Cluster')[cluster_features].mean().round(2)

group_stats

,age,balance,day,duration_minutes,campaign,previous,pdays
Cluster,,,,,,,
0,40.53,1437.53,13.83,3.90,2.12,3.54,259.95
1,41.31,1129.67,8.43,3.41,2.21,0.09,2.80
2,40.69,1480.84,22.89,3.19,2.55,0.08,3.88
3,41.28,1791.23,15.82,15.24,2.41,0.24,15.49
4,40.61,1144.60,21.95,2.39,15.84,0.02,0.05


| Cluster | Main characteristic                                  | Interpretation                           |
| ------- | ---------------------------------------------------- | ---------------------------------------- |
| **0**   | Highest `previous` = 3.54 and `pdays` = 259.95       | Previously contacted customers           |
| **1**   | Low `previous` = 0.09 and low `pdays` = 2.80         | Less previously contacted customers      |
| **2**   | High `balance` = 1480.84                             | Higher-balance customers                 |
| **3**   | Highest `balance` = 1791.23 and duration = 15.24 min | Highly engaged, higher-balance customers |
| **4**   | Highest `campaign` = 15.84                           | Frequently contacted customers           |


- **Group-wise Statistics**: After assigning cluster numbers to the original dataset, the records were grouped according to their cluster labels. Mean values of the selected numerical features were calculated for each group to understand their characteristics. The analysis identified five distinct customer segments with differences in previous contact history, account balance, call duration, and campaign contact frequency.